# Comparing VQS & VQE vs QAOA for RF Security (PSD Anomaly, RACH Detection)

This notebook compares **Variational Quantum Simulation (VQS)** and **Variational Quantum Eigensolver (VQE)** against **QAOA** on two tasks:

- **PSD anomaly scoring**: benign vs **fake-SSB** (spectral shift). Optionally, tone jammer.
- **RACH preamble detection**: matched vs mismatched, using **reference-normalized** correlation features.

All three methods share the same **Ising cost Hamiltonians** (linear $Z_i$ + nearest-neighbor $Z_iZ_{i+1}$) built from small reference sets. We report separability via histograms and simple **AUROC**.

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import numpy as base_np
import numpy.fft as fft
import matplotlib.pyplot as plt
print('PennyLane:', qml.__version__)

## Synthetic RF helpers (PSD & RACH features)

In [ ]:
def gen_ofdm_like(N=1024, subcarriers=(60,120,200), amps=(1.0,0.8,0.6), noise=0.05, fs=1.0):
    t = base_np.arange(N)/fs
    x = base_np.zeros(N, dtype=complex)
    for f,a in zip(subcarriers,amps):
        x += a * base_np.exp(1j*2*base_np.pi*f*t/N)
    x += noise * (base_np.random.randn(N) + 1j * base_np.random.randn(N))
    return x
def add_tone_jammer(x, tone_bin=150, amp=3.0):
    N=len(x); n=base_np.arange(N)
    return x + amp*base_np.exp(1j*2*base_np.pi*tone_bin*n/N)
def shift_sync(x, shift_bins=6):
    X=fft.fft(x); X=base_np.roll(X, shift_bins); return fft.ifft(X)
def psd_feature(x, n_bins=16):
    P=base_np.abs(fft.fft(x))**2; k=len(P)//n_bins
    P=P[:k*n_bins].reshape(n_bins,k).mean(axis=1); P=P/(P.max()+1e-12)
    return P.astype(float)
def zadoff_chu(u,Nzc):
    n=base_np.arange(Nzc); return base_np.exp(-1j*base_np.pi*u*n*(n+1)/Nzc)
def rach_burst(zc, delay=0, snr_db=12.0):
    N=len(zc); x=base_np.zeros(N,dtype=complex); x[delay:]=zc[:N-delay]
    snr=10**(snr_db/10); sp=base_np.mean(base_np.abs(x)**2)+1e-12
    npwr=sp/snr; noise=base_np.sqrt(npwr/2)*(base_np.random.randn(N)+1j*base_np.random.randn(N))
    return x+noise
def corr_feature_refnorm(rx, ref, n_bins=16, ref_scale=None):
    r = base_np.correlate(rx, ref, mode='same'); c = base_np.abs(r)
    if ref_scale is None:
        rref = base_np.correlate(ref, ref, mode='same'); ref_scale = float(base_np.max(base_np.abs(rref)))
    c = c/(ref_scale+1e-12); k=len(c)//n_bins
    return c[:k*n_bins].reshape(n_bins,k).mean(axis=1).astype(float)

## Ising Hamiltonians + circuits (shared across methods)

In [ ]:
def build_ising_H_from_stats(mean_vec, cov_mat, wires, kappa=1.0):
    H=0; mu = mean_vec - mean_vec.mean()
    for wi,a in zip(wires, mu):
        H = H + float(a)*qml.PauliZ(wi)
    for i in range(len(wires)-1):
        J = kappa*float(cov_mat[i, i+1])
        if abs(J)>1e-12:
            H = H + J*(qml.PauliZ(wires[i]) @ qml.PauliZ(wires[i+1]))
    return H

def angle_embed(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation='Y')

def ansatz_vqe(theta, L, wires):
    n=len(wires); idx=0
    for _ in range(L):
        for w in wires: qml.RZ(theta[idx], wires=w); idx+=1
        for w in wires: qml.RX(theta[idx], wires=w); idx+=1
        for i in range(n): qml.CNOT(wires=[wires[i], wires[(i+1)%n]])
        for i in range(n): qml.IsingZZ(0.25, wires=[wires[i], wires[(i+1)%n]])

def qaoa_block(gammas, betas, H, wires):
    for g,b in zip(gammas, betas):
        qml.qaoa.cost_layer(g, H)
        mixer_h = qml.qaoa.x_mixer(wires)
    qml.qaoa.mixer_layer(b, mixer_h)

def scalar(x):
    try: return float(x)
    except Exception: return float(np.array(x))

## Data builders (train/val splits)

In [ ]:
def build_psd_dataset(B=16, N=1024, n_train=64, n_val=16):
    def sample(kind, n):
        S=[]
        for _ in range(n):
            x=gen_ofdm_like(N=N)
            if kind=='fake': x=shift_sync(x,6)
            S.append(psd_feature(x, n_bins=B))
        return base_np.stack(S,axis=0)
    S_b_tr=sample('benign', n_train); S_b_val=sample('benign', n_val)
    S_f_tr=sample('fake', n_train);  S_f_val=sample('fake', n_val)
    mu_b, mu_f = S_b_tr.mean(0), S_f_tr.mean(0)
    Cb, Cf = base_np.cov(S_b_tr.T), base_np.cov(S_f_tr.T)
    return (S_b_tr,S_f_tr,S_b_val,S_f_val, mu_b,mu_f,Cb,Cf)

def build_rach_dataset(B=16, Nzc=839, u_root=167, n_pos=21, n_neg=28):
    zc=zadoff_chu(u_root, Nzc)
    rref=base_np.correlate(zc,zc,mode='same'); ref_scale=float(base_np.max(base_np.abs(rref)))
    R_pos=base_np.stack([corr_feature_refnorm(rach_burst(zc, 12+d, 12.0), zc, n_bins=B, ref_scale=ref_scale) for d in range(-3,4)],axis=0)
    R_neg=base_np.stack([corr_feature_refnorm(rach_burst(zc, base_np.random.randint(40,120), 8.0), zc, n_bins=B, ref_scale=ref_scale) for _ in range(n_neg)],axis=0)
    # Split
    Rp_tr, Rp_val = R_pos[:n_pos//3*2], R_pos[n_pos//3*2:]
    Rn_tr, Rn_val = R_neg[:n_neg//3*2], R_neg[n_neg//3*2:]
    mu_p, mu_n = Rp_tr.mean(0), Rn_tr.mean(0)
    Cp, Cn = base_np.cov(Rp_tr.T), base_np.cov(Rn_tr.T)
    return (zc, ref_scale, Rp_tr, Rn_tr, Rp_val, Rn_val, mu_p,mu_n,Cp,Cn)

## AUROC utility (no sklearn)

In [ ]:
def auroc(scores_pos, scores_neg):
    # scores_pos assumed higher for positive class
    s = base_np.concatenate([scores_pos, scores_neg])
    y = base_np.concatenate([base_np.ones_like(scores_pos), base_np.zeros_like(scores_neg)])
    order = base_np.argsort(s)
    s, y = s[order], y[order]
    # thresholds: unique scores
    uniq = base_np.unique(s)
    tpr=[]; fpr=[]
    P = y.sum(); N = len(y)-P
    for t in uniq:
        pred = (s>=t).astype(int)
        TP = (pred*y).sum(); FP = (pred*(1-y)).sum()
        tpr.append(TP/(P+1e-12)); fpr.append(FP/(N+1e-12))
    # sort by fpr and integrate
    order = base_np.argsort(fpr)
    fpr = base_np.array(fpr)[order]; tpr = base_np.array(tpr)[order]
    auc = base_np.trapz(tpr, fpr)
    return float(auc), fpr, tpr

# Part A — PSD: VQS vs VQE vs QAOA

In [ ]:
base_np.random.seed(7)
B=16; wires=list(range(B))
S_b_tr,S_f_tr,S_b_val,S_f_val, mu_b,mu_f, Cb,Cf = build_psd_dataset(B=B)
H_b = build_ising_H_from_stats(mu_b,Cb,wires); H_f = build_ising_H_from_stats(mu_f,Cf,wires)

# --- VQS (McLachlan, diagonal metric) ---
dev_vqs = qml.device('default.qubit', wires=B)
np.random.seed(0)
theta_vqs = np.random.uniform(-0.1,0.1, size=2*B*2)
theta_vqs = np.array(theta_vqs, requires_grad=True)

def vqs_step(features, theta, H, dt=0.05, L=2, lam=1e-3, diag=True):
    @qml.qnode(dev_vqs)
    def expH(th): angle_embed(features,wires); ansatz_vqe(th,L,wires); return qml.expval(H)
    M = np.diag(qml.metric_tensor(expH, approx='diag')(theta)) if diag else qml.metric_tensor(expH)(theta)
    V = qml.grad(expH)(theta)
    thdot = np.linalg.solve(M + lam*np.eye(len(M)), V)
    return theta + dt*thdot, float(expH(theta))

def vqs_score_batch(Xb, Xf, steps=16):
    th = theta_vqs.copy()
    # warm-up on benign mean
    xb_mean = S_b_tr.mean(axis=0)
    for _ in range(8): th,_ = vqs_step(xb_mean, th, H_b)
    def dE(x):
        # one benign step then contrastive
        th_loc,_ = vqs_step(x, th, H_b)
        @qml.qnode(dev_vqs)
        def eH(th, H): angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
        Eb = float(eH(th_loc, H_b)); Ef = float(eH(th_loc, H_f))
        return Ef - Eb
    s_b = base_np.array([dE(x) for x in Xb]); s_f = base_np.array([dE(x) for x in Xf])
    return s_b, s_f

psd_vqs_b, psd_vqs_f = vqs_score_batch(S_b_val, S_f_val)
auc_vqs, fpr_vqs, tpr_vqs = auroc(psd_vqs_f, psd_vqs_b)
print('PSD VQS AUROC:', round(auc_vqs,4))

# --- VQE (contrastive hinge training) ---
dev_vqe = qml.device('default.qubit', wires=B)
@qml.qnode(dev_vqe)
def energy_vqe(th, x, H):
    angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
def batch_energy_vqe(th, X, H):
    return np.mean(np.stack([energy_vqe(th, x, H) for x in X]))
def loss_vqe(th, Xb, Xf, margin=0.15, lam=1.0):
    Eb = batch_energy_vqe(th, Xb, H_b); Ef = batch_energy_vqe(th, Xf, H_f)
    return Eb + lam * qml.math.maximum(0.0, margin - (Ef - Eb))
np.random.seed(1)
theta_vqe = np.random.uniform(-0.1,0.1, size=2*B*2)
opt = qml.AdamOptimizer(stepsize=0.05)
for step in range(60):
    i=(step*8)%48; Xb=S_b_tr[i:i+8]; Xf=S_f_tr[i:i+8]
    theta_vqe,_ = opt.step_and_cost(lambda th: loss_vqe(th,Xb,Xf), theta_vqe)
def deltaE_vqe(x):
    return float(energy_vqe(theta_vqe, x, H_f) - energy_vqe(theta_vqe, x, H_b))
psd_vqe_b = base_np.array([deltaE_vqe(x) for x in S_b_val])
psd_vqe_f = base_np.array([deltaE_vqe(x) for x in S_f_val])
auc_vqe, fpr_vqe, tpr_vqe = auroc(psd_vqe_f, psd_vqe_b)
print('PSD VQE AUROC:', round(auc_vqe,4))

# --- QAOA (p=2) ---
dev_q = qml.device('default.qubit', wires=B); p=2
@qml.qnode(dev_q)
def energy_qaoa(params, x, H):
    gam=params[:p]; bet=params[p:2*p]
    angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
def batch_energy_qaoa(pr, X, H):
    return np.mean(np.stack([energy_qaoa(pr, x, H) for x in X]))
def loss_qaoa(pr, Xb, Xf, margin=0.15, lam=1.0):
    Eb=batch_energy_qaoa(pr,Xb,H_b); Ef=batch_energy_qaoa(pr,Xf,H_f)
    return Eb + lam * qml.math.maximum(0.0, margin - (Ef - Eb))
np.random.seed(2)
params_q = np.random.uniform(-0.1,0.1,size=2*p)
optq = qml.AdamOptimizer(stepsize=0.05)
for step in range(60):
    i=(step*8)%48; Xb=S_b_tr[i:i+8]; Xf=S_f_tr[i:i+8]
    params_q,_ = optq.step_and_cost(lambda pr: loss_qaoa(pr,Xb,Xf), params_q)
def deltaE_qaoa(x):
    return float(energy_qaoa(params_q, x, H_f) - energy_qaoa(params_q, x, H_b))
psd_q_b = base_np.array([deltaE_qaoa(x) for x in S_b_val])
psd_q_f = base_np.array([deltaE_qaoa(x) for x in S_f_val])
auc_q, fpr_q, tpr_q = auroc(psd_q_f, psd_q_b)
print('PSD QAOA AUROC:', round(auc_q,4))

# --- Plots ---
plt.figure(figsize=(6,4))
plt.hist([psd_vqs_b, psd_vqs_f], bins=15, alpha=0.6, label=['benign','fake'])
plt.title('PSD ΔE (VQS)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()
plt.figure(figsize=(6,4))
plt.hist([psd_vqe_b, psd_vqe_f], bins=15, alpha=0.6, label=['benign','fake'])
plt.title('PSD ΔE (VQE)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()
plt.figure(figsize=(6,4))
plt.hist([psd_q_b, psd_q_f], bins=15, alpha=0.6, label=['benign','fake'])
plt.title('PSD ΔE (QAOA)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(fpr_vqs, tpr_vqs, label=f'VQS AUC={auc_vqs:.2f}')
plt.plot(fpr_vqe, tpr_vqe, label=f'VQE AUC={auc_vqe:.2f}')
plt.plot(fpr_q,   tpr_q,   label=f'QAOA AUC={auc_q:.2f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('PSD ROC (fake vs benign)')
plt.legend(); plt.show()

# Part B — RACH: VQS vs VQE vs QAOA (reference-normalized features)

In [ ]:
base_np.random.seed(11)
B2=16; wires2=list(range(B2))
zc, ref_scale, Rp_tr, Rn_tr, Rp_val, Rn_val, mu_p,mu_n,Cp,Cn = build_rach_dataset(B=B2)
H_pos = build_ising_H_from_stats(mu_p,Cp,wires2); H_neg = build_ising_H_from_stats(mu_n,Cn,wires2)

# --- VQS ---
dev_vqs2 = qml.device('default.qubit', wires=B2)
np.random.seed(3)
theta_vqs2 = np.random.uniform(-0.1,0.1, size=2*B2*2); theta_vqs2 = np.array(theta_vqs2, requires_grad=True)
def vqs_step2(features, theta, H, dt=0.05, L=2, lam=1e-3, diag=True):
    @qml.qnode(dev_vqs2)
    def expH(th): angle_embed(features,wires2); ansatz_vqe(th,L,wires2); return qml.expval(H)
    M = np.diag(qml.metric_tensor(expH, approx='diag')(theta)) if diag else qml.metric_tensor(expH)(theta)
    V = qml.grad(expH)(theta)
    thdot = np.linalg.solve(M + lam*np.eye(len(M)), V)
    return theta + dt*thdot, float(expH(theta))
def rach_vqs_scores(Rp, Rn):
    th = theta_vqs2.copy()
    xref = Rp_tr.mean(axis=0)
    for _ in range(8): th,_ = vqs_step2(xref, th, H_pos)
    def dE(x):
        th_loc,_ = vqs_step2(x, th, H_pos)
        @qml.qnode(dev_vqs2)
        def eH(th, H): angle_embed(x,wires2); ansatz_vqe(th,2,wires2); return qml.expval(H)
        Ep=float(eH(th_loc, H_pos)); En=float(eH(th_loc, H_neg)); return En-Ep
    s_pos=base_np.array([dE(x) for x in Rp]); s_neg=base_np.array([dE(x) for x in Rn]); return s_pos, s_neg
rach_vqs_pos, rach_vqs_neg = rach_vqs_scores(Rp_val, Rn_val)
auc_vqs_r, fpr_vqs_r, tpr_vqs_r = auroc(rach_vqs_neg, rach_vqs_pos)  # higher -> mismatch (neg)
print('RACH VQS AUROC (neg vs pos):', round(auc_vqs_r,4))

# --- VQE ---
dev_vqe2 = qml.device('default.qubit', wires=B2)
@qml.qnode(dev_vqe2)
def energy_vqe2(th, x, H): angle_embed(x,wires2); ansatz_vqe(th,2,wires2); return qml.expval(H)
def batch_energy_vqe2(th, X, H): return np.mean(np.stack([energy_vqe2(th, x, H) for x in X]))
def loss_vqe2(th, Rp, Rn, margin=0.08, lam=1.0):
    Ep=batch_energy_vqe2(th, Rp, H_pos); En=batch_energy_vqe2(th, Rn, H_neg)
    return Ep + lam * qml.math.maximum(0.0, margin - (En - Ep))
np.random.seed(4)
theta_vqe2 = np.random.uniform(-0.1,0.1, size=2*B2*2)
opt2 = qml.AdamOptimizer(stepsize=0.05)
for step in range(60):
    i=(step*6)%len(Rp_tr); RpB=Rp_tr[max(0,i-3):i+4]; RnB=Rn_tr[i:i+7]
    theta_vqe2,_ = opt2.step_and_cost(lambda th: loss_vqe2(th,RpB,RnB), theta_vqe2)
def dE_vqe2(x): return float(energy_vqe2(theta_vqe2, x, H_neg) - energy_vqe2(theta_vqe2, x, H_pos))
rach_vqe_pos=base_np.array([dE_vqe2(x) for x in Rp_val]); rach_vqe_neg=base_np.array([dE_vqe2(x) for x in Rn_val])
auc_vqe_r, fpr_vqe_r, tpr_vqe_r = auroc(rach_vqe_neg, rach_vqe_pos)
print('RACH VQE AUROC (neg vs pos):', round(auc_vqe_r,4))

# --- QAOA ---
dev_q2 = qml.device('default.qubit', wires=B2); pR=2
@qml.qnode(dev_q2)
def energy_qaoa2(pr, x, H):
    gam=pr[:pR]; bet=pr[pR:2*pR]
    angle_embed(x,wires2); qaoa_block(gam,bet,H,wires2); return qml.expval(H)
def batch_energy_qaoa2(pr, X, H): return np.mean(np.stack([energy_qaoa2(pr,x,H) for x in X]))
def loss_qaoa2(pr, Rp, Rn, margin=0.08, lam=1.0):
    Ep=batch_energy_qaoa2(pr,Rp,H_pos); En=batch_energy_qaoa2(pr,Rn,H_neg)
    return Ep + lam * qml.math.maximum(0.0, margin - (En - Ep))
np.random.seed(5)
params_r = np.random.uniform(-0.1,0.1, size=2*pR)
optq2 = qml.AdamOptimizer(stepsize=0.05)
for step in range(60):
    i=(step*6)%len(Rp_tr); RpB=Rp_tr[max(0,i-3):i+4]; RnB=Rn_tr[i:i+7]
    params_r,_ = optq2.step_and_cost(lambda pr: loss_qaoa2(pr,RpB,RnB), params_r)
def dE_qaoa2(x): return float(energy_qaoa2(params_r, x, H_neg) - energy_qaoa2(params_r, x, H_pos))
rach_q_pos=base_np.array([dE_qaoa2(x) for x in Rp_val]); rach_q_neg=base_np.array([dE_qaoa2(x) for x in Rn_val])
auc_q_r, fpr_q_r, tpr_q_r = auroc(rach_q_neg, rach_q_pos)
print('RACH QAOA AUROC (neg vs pos):', round(auc_q_r,4))

# --- Plots ---
plt.figure(figsize=(6,4))
plt.hist([rach_vqs_pos, rach_vqs_neg], bins=15, alpha=0.6, label=['pos(match)','neg(mismatch)'])
plt.title('RACH ΔE (VQS)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()
plt.figure(figsize=(6,4))
plt.hist([rach_vqe_pos, rach_vqe_neg], bins=15, alpha=0.6, label=['pos(match)','neg(mismatch)'])
plt.title('RACH ΔE (VQE)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()
plt.figure(figsize=(6,4))
plt.hist([rach_q_pos, rach_q_neg], bins=15, alpha=0.6, label=['pos(match)','neg(mismatch)'])
plt.title('RACH ΔE (QAOA)'); plt.xlabel('ΔE'); plt.ylabel('Count'); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(fpr_vqs_r, tpr_vqs_r, label=f'VQS AUC={auc_vqs_r:.2f}')
plt.plot(fpr_vqe_r, tpr_vqe_r, label=f'VQE AUC={auc_vqe_r:.2f}')
plt.plot(fpr_q_r,   tpr_q_r,   label=f'QAOA AUC={auc_q_r:.2f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('RACH ROC (neg vs pos)')
plt.legend(); plt.show()

## Notes
- For speed, training loops use small iterations; increase steps for stronger separation.
- Toggle VQS full metric by replacing `approx='diag'` with full `qml.metric_tensor`.
- Replace synthetic features with your capture pipeline to test on real data.
- You can add tone-jammer heads for PSD by building `H_tone` from a tone reference set and scoring with `ΔE_tone`.

## Extensions: Tone-Jammer Head, Multi-Root RACH, and SPSA + Shot Noise

This section augments the comparison with (i) a tone-jammer PSD head, (ii) a **multi-root** RACH head bank, and (iii) an SPSA training option with finite shots for hardware-style noise.

In [ ]:
def build_psd_tone_dataset(B=16, N=1024, n_train=64, n_val=16):
    def sample(kind, n):
        S=[]
        for _ in range(n):
            x=gen_ofdm_like(N=N)
            if kind=='tone': x=add_tone_jammer(x, tone_bin=150, amp=3.0)
            S.append(psd_feature(x, n_bins=B))
        return base_np.stack(S,axis=0)
    S_b_tr=sample('benign', n_train); S_b_val=sample('benign', n_val)
    S_t_tr=sample('tone',   n_train); S_t_val=sample('tone',   n_val)
    mu_b, mu_t = S_b_tr.mean(0), S_t_tr.mean(0)
    Cb, Ct = base_np.cov(S_b_tr.T), base_np.cov(S_t_tr.T)
    return (S_b_tr,S_t_tr,S_b_val,S_t_val, mu_b,mu_t,Cb,Ct)

# Build tone set and Hamiltonians
S_b_tr_t,S_t_tr,S_b_val_t,S_t_val, mu_b_t,mu_t, Cb_t,Ct = build_psd_tone_dataset(B=B)
H_b_t = build_ising_H_from_stats(mu_b_t, Cb_t, wires)
H_t   = build_ising_H_from_stats(mu_t,   Ct,   wires)

# VQS tone scoring
def vqs_score_batch_tone(Xb, Xt, steps=16):
    dev_tmp = qml.device('default.qubit', wires=B)
    np.random.seed(10)
    th0 = np.random.uniform(-0.1,0.1, size=2*B*2); th0 = np.array(th0, requires_grad=True)
    @qml.qnode(dev_tmp)
    def expH_t(th, x, H): angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
    def step(x, th, H, dt=0.05, lam=1e-3):
        def f(th): return expH_t(th, x, H)
        M = np.diag(qml.metric_tensor(f, approx='diag')(th))
        V = qml.grad(f)(th)
        thdot = np.linalg.solve(M + lam*np.eye(len(M)), V)
        return th + dt*thdot
    th=th0.copy(); xb_mean = S_b_tr_t.mean(axis=0)
    for _ in range(8): th = step(xb_mean, th, H_b_t)
    def dE(x):
        th_loc = step(x, th, H_b_t)
        Eb=float(expH_t(th_loc, x, H_b_t)); Et=float(expH_t(th_loc, x, H_t)); return Et-Eb
    s_b = base_np.array([dE(x) for x in Xb]); s_t = base_np.array([dE(x) for x in Xt])
    return s_b, s_t

psd_vqs_b_t, psd_vqs_t = vqs_score_batch_tone(S_b_val_t, S_t_val)
auc_vqs_t, fpr_vqs_t, tpr_vqs_t = auroc(psd_vqs_t, psd_vqs_b_t)
print('PSD (tone) VQS AUROC:', round(auc_vqs_t,4))

# VQE tone scoring (reuse trained theta_vqe from fake or retrain quickly on tone)
def train_vqe_tone(steps=40):
    dev_tmp = qml.device('default.qubit', wires=B)
    @qml.qnode(dev_tmp)
    def eV(th, x, H): angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
    def L(th, Xb, Xt, margin=0.15):
        Eb = np.mean(np.stack([eV(th,x,H_b_t) for x in Xb]))
        Et = np.mean(np.stack([eV(th,x,H_t) for x in Xt]))
        return Eb + qml.math.maximum(0.0, margin - (Et - Eb))
    np.random.seed(11)
    th = np.random.uniform(-0.1,0.1, size=2*B*2)
    opt = qml.AdamOptimizer(stepsize=0.05)
    for s in range(steps):
        i=(s*8)%48; Xb=S_b_tr_t[i:i+8]; Xt=S_t_tr[i:i+8]
        th,_ = opt.step_and_cost(lambda t: L(t,Xb,Xt), th)
    return th, eV
theta_vqe_t, eV_t = train_vqe_tone()
def dE_vqe_t(x): return float(eV_t(theta_vqe_t, x, H_t) - eV_t(theta_vqe_t, x, H_b_t))
psd_vqe_b_t = base_np.array([dE_vqe_t(x) for x in S_b_val_t])
psd_vqe_t   = base_np.array([dE_vqe_t(x) for x in S_t_val])
auc_vqe_t, fpr_vqe_t, tpr_vqe_t = auroc(psd_vqe_t, psd_vqe_b_t)
print('PSD (tone) VQE AUROC:', round(auc_vqe_t,4))

# QAOA tone scoring
dev_q_t = qml.device('default.qubit', wires=B); p_t=2
@qml.qnode(dev_q_t)
def eQ(params, x, H):
    gam=params[:p_t]; bet=params[p_t:2*p_t]
    angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
def Lq(params, Xb, Xt, margin=0.15):
    Eb = np.mean(np.stack([eQ(params,x,H_b_t) for x in Xb]))
    Et = np.mean(np.stack([eQ(params,x,H_t) for x in Xt]))
    return Eb + qml.math.maximum(0.0, margin - (Et - Eb))
np.random.seed(12)
params_t = np.random.uniform(-0.1,0.1, size=2*p_t)
opt_t = qml.AdamOptimizer(stepsize=0.05)
for s in range(40):
    i=(s*8)%48; Xb=S_b_tr_t[i:i+8]; Xt=S_t_tr[i:i+8]
    params_t,_ = opt_t.step_and_cost(lambda pr: Lq(pr,Xb,Xt), params_t)
def dE_q_t(x): return float(eQ(params_t, x, H_t) - eQ(params_t, x, H_b_t))
psd_q_b_t = base_np.array([dE_q_t(x) for x in S_b_val_t])
psd_q_t   = base_np.array([dE_q_t(x) for x in S_t_val])
auc_q_t, fpr_q_t, tpr_q_t = auroc(psd_q_t, psd_q_b_t)
print('PSD (tone) QAOA AUROC:', round(auc_q_t,4))

plt.figure(figsize=(6,4)); plt.plot(fpr_vqs_t, tpr_vqs_t, label=f'VQS {auc_vqs_t:.2f}')
plt.plot(fpr_vqe_t, tpr_vqe_t, label=f'VQE {auc_vqe_t:.2f}')
plt.plot(fpr_q_t,   tpr_q_t,   label=f'QAOA {auc_q_t:.2f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('PSD ROC (tone vs benign)'); plt.legend(); plt.show()

### Multi-root RACH head bank
We build positive heads $H_{\text{pos}}^{(c)}$ for several ZC roots and a shared negative head $H_{\text{neg}}$.
Prediction uses $\arg\min_c \Delta E_c(x)$ with $\Delta E_c = \langle H_{\text{neg}}\rangle-\langle H_{\text{pos}}^{(c)}\rangle$.

In [ ]:
def build_rach_multiroot_dataset(roots=(167,173,181), B=16, Nzc=839, snr_pos=12.0, snr_neg=8.0):
    datasets = {}
    all_neg = []
    for u in roots:
        zc = zadoff_chu(u, Nzc)
        rref = base_np.correlate(zc,zc,mode='same'); ref_scale = float(base_np.max(base_np.abs(rref)))
        pos = base_np.stack([corr_feature_refnorm(rach_burst(zc, delay=12+d, snr_db=snr_pos), zc, n_bins=B, ref_scale=ref_scale)
                             for d in range(-3,4)], axis=0)
        # Negatives: wrong delays + wrong roots mix
        neg1 = base_np.stack([corr_feature_refnorm(rach_burst(zc, delay=base_np.random.randint(40,120), snr_db=snr_neg), zc, n_bins=B, ref_scale=ref_scale)
                              for _ in range(24)], axis=0)
        datasets[u] = {'pos': pos, 'neg_local': neg1, 'zc': zc, 'ref_scale': ref_scale}
        all_neg.append(neg1)
    # Cross-root negatives
    roots_list = list(roots)
    for i,u in enumerate(roots_list):
        zc_u = datasets[u]['zc']; ref_u = datasets[u]['ref_scale']
        others = [r for r in roots_list if r!=u]
        cross = []
        for v in others:
            zc_v = datasets[v]['zc']
            cross.append(base_np.stack([
                corr_feature_refnorm(rach_burst(zc_v, delay=12+d, snr_db=snr_pos), zc_u, n_bins=B, ref_scale=ref_u)
                for d in range(-3,4)
            ], axis=0))
        if len(cross)>0:
            cross = base_np.concatenate(cross, axis=0)
            datasets[u]['neg_cross'] = cross
            all_neg.append(cross)
    # Shared negative pool
    shared_neg = base_np.concatenate(all_neg, axis=0)
    return datasets, shared_neg

roots = (167,173,181)
Bmr = 16; wires_mr = list(range(Bmr))
ds_mr, shared_neg = build_rach_multiroot_dataset(roots=roots, B=Bmr)

# Build heads H_pos^(c) and shared H_neg
Hpos_bank = {}
for u in roots:
    pos = ds_mr[u]['pos']
    mu = pos.mean(axis=0); C = base_np.cov(pos.T)
    Hpos_bank[u] = build_ising_H_from_stats(mu, C, wires_mr)
mu_n = shared_neg.mean(axis=0); Cn = base_np.cov(shared_neg.T)
Hneg_bank = build_ising_H_from_stats(mu_n, Cn, wires_mr)

# Define scorers for VQE and QAOA banks (to keep runtime reasonable)
dev_b_vqe = qml.device('default.qubit', wires=Bmr)
@qml.qnode(dev_b_vqe)
def eV_bank(th, x, H): angle_embed(x,wires_mr); ansatz_vqe(th,2,wires_mr); return qml.expval(H)
def train_vqe_bank(pos_pool, neg_pool, steps=50):
    th = np.random.uniform(-0.1,0.1, size=2*Bmr*2)
    opt = qml.AdamOptimizer(stepsize=0.05)
    pos_concat = base_np.concatenate([ds_mr[u]['pos'] for u in roots], axis=0)
    for s in range(steps):
        i=(s*8)% (len(pos_concat)-8)
        Xp = pos_concat[i:i+8]
        Xn = neg_pool[i% (len(neg_pool)-8): i% (len(neg_pool)-8)+8]
        def L(th):
            Ep = np.mean(np.stack([eV_bank(th,x,Hpos_bank[roots[0]]) for x in Xp]))  # proxy: one head for training
            En = np.mean(np.stack([eV_bank(th,x,Hneg_bank)          for x in Xn]))
            return Ep + qml.math.maximum(0.0, 0.08 - (En - Ep))
        th,_ = opt.step_and_cost(L, th)
    return th

theta_bank = train_vqe_bank(ds_mr, shared_neg, steps=40)
def deltaE_bank_vqe(x):
    scores = {}
    for u in roots:
        Ep = eV_bank(theta_bank, x, Hpos_bank[u])
        En = eV_bank(theta_bank, x, Hneg_bank)
        scores[u] = float(En - Ep)
    pred_root = min(scores, key=scores.get)
    min_score = scores[pred_root]
    return pred_root, min_score, scores

# Evaluate root classification accuracy on held-out positives
correct=0; total=0; mins=[]
for u in roots:
    for x in ds_mr[u]['pos']:
        pred, smin, _ = deltaE_bank_vqe(x)
        correct += int(pred==u); total+=1; mins.append(smin)
acc_root = correct/total
print('Multi-root RACH (VQE bank) root-accuracy:', round(acc_root,3))

### SPSA training with finite shots (hardware-style) for QAOA (PSD)
We train QAOA angles using SPSA and `shots=1000` to emulate sampling noise.

In [ ]:
SHOT_NOISE=True
shots = 1000 if SHOT_NOISE else None
dev_spsa = qml.device('default.qubit', wires=B, shots=shots)
@qml.qnode(dev_spsa)
def energy_qaoa_shot(params, x, H):
    p=2; gam=params[:p]; bet=params[p:2*p]
    angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
def batch_energy_shot(params, X, H):
    return np.mean(np.stack([energy_qaoa_shot(params, x, H) for x in X]))
def loss_shot(params, Xb, Xf, margin=0.15, lam=1.0):
    Eb=batch_energy_shot(params,Xb,H_b); Ef=batch_energy_shot(params,Xf,H_f)
    return Eb + lam*qml.math.maximum(0.0, margin-(Ef-Eb))

def spsa_step(params, loss_fn, a=0.05, c=0.1):
    # Two-point SPSA gradient estimate
    dim=len(params); delta = 2*base_np.random.randint(0,2,size=dim)-1  # ±1
    delta = np.array(delta, dtype=float)
    Lp = loss_fn(params + c*delta)
    Lm = loss_fn(params - c*delta)
    ghat = (Lp - Lm)/(2*c*delta)
    return params - a*ghat

np.random.seed(21)
params_spsa = np.random.uniform(-0.1,0.1, size=4)  # 2p with p=2
for s in range(40):
    i=(s*8)%48; Xb=S_b_tr[i:i+8]; Xf=S_f_tr[i:i+8]
    params_spsa = spsa_step(params_spsa, lambda pr: loss_shot(pr,Xb,Xf))
def deltaE_spsa(x): return float(energy_qaoa_shot(params_spsa, x, H_f) - energy_qaoa_shot(params_spsa, x, H_b))
psd_spsa_b = base_np.array([deltaE_spsa(x) for x in S_b_val])
psd_spsa_f = base_np.array([deltaE_spsa(x) for x in S_f_val])
auc_spsa, fpr_spsa, tpr_spsa = auroc(psd_spsa_f, psd_spsa_b)
print('PSD QAOA (SPSA, shots=1000) AUROC:', round(auc_spsa,4))
plt.figure(figsize=(6,4))
plt.plot(fpr_spsa, tpr_spsa, label=f'SPSA shots AUC={auc_spsa:.2f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('PSD ROC (QAOA SPSA shots)'); plt.legend(); plt.show()

## Multi-root RACH: Confusion Matrix and One-vs-Rest ROC (VQE bank)

We evaluate the VQE-trained head bank by:
1. Building a **confusion matrix** over held-out positives for each root, using prediction \(\hat{c}=\arg\min_c \Delta E_c\) where \(\Delta E_c=\langle H_{\text{neg}}\rangle-\langle H_{\text{pos}}^{(c)}\rangle\).
2. Plotting **one-vs-rest ROC** curves per root with score \(s_c(x)=-\Delta E_c(x)\) (higher means “more like root \(c\)”).

In [ ]:
import matplotlib.pyplot as plt
import numpy as _np

# Ensure required symbols exist
roots_list = list(roots)
idx_of = {u:i for i,u in enumerate(roots_list)}

def deltaE_scores_all(x):
    scores = {}
    for u in roots_list:
        Ep = eV_bank(theta_bank, x, Hpos_bank[u])
        En = eV_bank(theta_bank, x, Hneg_bank)
        scores[u] = float(En - Ep)
    return scores

# 1) Confusion matrix over positives
C = _np.zeros((len(roots_list), len(roots_list)), dtype=int)
for u_true in roots_list:
    for x in ds_mr[u_true]['pos']:
        scores = deltaE_scores_all(x)
        u_pred = min(scores, key=scores.get)
        C[idx_of[u_true], idx_of[u_pred]] += 1

plt.figure(figsize=(5,4))
plt.imshow(C, interpolation='nearest')
plt.title('RACH multi-root confusion (VQE bank)')
plt.xlabel('Predicted root')
plt.ylabel('True root')
plt.xticks(range(len(roots_list)), roots_list)
plt.yticks(range(len(roots_list)), roots_list)
for i in range(C.shape[0]):
    for j in range(C.shape[1]):
        plt.text(j, i, str(C[i,j]), ha='center', va='center')
plt.tight_layout(); plt.show()

acc = _np.trace(C)/C.sum()
print({'root_accuracy': float(acc)})

# 2) One-vs-rest ROC per root
def collect_scores_one_vs_rest(u_pos):
    # positives: ds_mr[u_pos]['pos']
    # negatives: all other roots' pos + shared_neg
    pos_scores = []
    neg_scores = []
    for x in ds_mr[u_pos]['pos']:
        s = deltaE_scores_all(x)[u_pos]
        pos_scores.append(-s)  # higher should indicate class u_pos
    for u_other in roots_list:
        if u_other==u_pos: continue
        for x in ds_mr[u_other]['pos']:
            s = deltaE_scores_all(x)[u_pos]
            neg_scores.append(-s)
    # also include a sample from shared negatives (limit to keep runtime modest)
    for x in shared_neg[:min(64, len(shared_neg))]:
        s = deltaE_scores_all(x)[u_pos]
        neg_scores.append(-s)
    return _np.array(pos_scores, dtype=float), _np.array(neg_scores, dtype=float)

plt.figure(figsize=(6,4))
legend_items = []
for u in roots_list:
    sp, sn = collect_scores_one_vs_rest(u)
    auc, fpr, tpr = auroc(sp, sn)
    plt.plot(fpr, tpr, label=f'root {u} AUC={auc:.2f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('RACH multi-root: one-vs-rest ROC (VQE bank)')
plt.legend(); plt.show()

## Multi-root RACH: QAOA Head Bank (Confusion Matrix + One-vs-Rest ROC)

We mirror the VQE-bank analysis using a **single set of QAOA angles** shared across roots. The loss minimizes the
mean positive-head energy across all roots while enforcing a margin against a shared negative head:
$$\mathcal{L}(\gamma,\beta)=\mathbb{E}_{(c,x)\in \text{pos bank}} \langle H_{\text{pos}}^{(c)}\rangle\;+
\;\max\big(0, m - (\mathbb{E}_{x\in \text{neg}}\langle H_{\text{neg}}\rangle - \mathbb{E}_{(c,x)\in \text{pos bank}}\langle H_{\text{pos}}^{(c)}\rangle)\big).$$
Prediction uses $\hat{c}=\arg\min_c \Delta E_c(x)$ with $\Delta E_c=\langle H_{\text{neg}}\rangle-\langle H_{\text{pos}}^{(c)}\rangle$.

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import numpy as _np
import matplotlib.pyplot as plt

# Device and QAOA energy for a given head
p_bank = 2
dev_b_qaoa = qml.device('default.qubit', wires=Bmr)
@qml.qnode(dev_b_qaoa)
def eQ_bank(params, x, H):
    gam = params[:p_bank]; bet = params[p_bank:2*p_bank]
    angle_embed(x, wires_mr)
    qaoa_block(gam, bet, H, wires_mr)
    return qml.expval(H)

def batch_pos_energy(params, pos_batches):
    # pos_batches: list of (u_root, x_feature)
    vals = []
    for u,x in pos_batches:
        vals.append(eQ_bank(params, x, Hpos_bank[u]))
    return np.mean(np.stack(vals))

def batch_neg_energy(params, Xneg):
    return np.mean(np.stack([eQ_bank(params, x, Hneg_bank) for x in Xneg]))

def sample_pos_minibatch(size=12):
    # roughly balanced over roots
    items = []
    each = max(1, size // len(roots))
    for u in roots:
        pos = ds_mr[u]['pos']
        idx = _np.random.choice(len(pos), size=each, replace=len(pos)<each)
        for j in idx:
            items.append((u, pos[j]))
    # pad if needed
    while len(items) < size:
        u = _np.random.choice(list(roots))
        j = _np.random.randint(len(ds_mr[u]['pos']))
        items.append((u, ds_mr[u]['pos'][j]))
    return items[:size]

def sample_neg_minibatch(size=12):
    idx = _np.random.choice(len(shared_neg), size=size, replace=len(shared_neg)<size)
    return shared_neg[idx]

def loss_qaoa_bank(params, m=0.08):
    pos_mb = sample_pos_minibatch(size=12)
    neg_mb = sample_neg_minibatch(size=12)
    Ep = batch_pos_energy(params, pos_mb)
    En = batch_neg_energy(params, neg_mb)
    gap = En - Ep
    return Ep + qml.math.maximum(0.0, m - gap)

# Train shared angles
np.random.seed(42)
params_bank = np.random.uniform(low=-0.1, high=0.1, size=2*p_bank)
opt_bank = qml.AdamOptimizer(stepsize=0.05)
loss_trace_bank = []
for step in range(60):
    params_bank, curr = opt_bank.step_and_cost(loss_qaoa_bank, params_bank)
    loss_trace_bank.append(float(curr))
print('QAOA bank final loss:', round(loss_trace_bank[-1],4))
plt.figure(figsize=(6,4)); plt.plot(loss_trace_bank); plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('QAOA bank training'); plt.show()

def deltaE_scores_all_qaoa(x):
    scores = {}
    for u in roots:
        Ep = eQ_bank(params_bank, x, Hpos_bank[u])
        En = eQ_bank(params_bank, x, Hneg_bank)
        scores[u] = float(En - Ep)
    return scores

# Confusion matrix over positives
Cq = _np.zeros((len(roots), len(roots)), dtype=int)
for u_true in roots:
    for x in ds_mr[u_true]['pos']:
        scores = deltaE_scores_all_qaoa(x)
        u_pred = min(scores, key=scores.get)
        Cq[list(roots).index(u_true), list(roots).index(u_pred)] += 1
plt.figure(figsize=(5,4))
plt.imshow(Cq, interpolation='nearest')
plt.title('RACH multi-root confusion (QAOA bank)')
plt.xlabel('Predicted root'); plt.ylabel('True root')
plt.xticks(range(len(roots)), list(roots)); plt.yticks(range(len(roots)), list(roots))
for i in range(Cq.shape[0]):
    for j in range(Cq.shape[1]):
        plt.text(j, i, str(Cq[i,j]), ha='center', va='center')
plt.tight_layout(); plt.show()
acc_q = _np.trace(Cq)/Cq.sum(); print({'root_accuracy_qaoa': float(acc_q)})

# One-vs-rest ROC per root
plt.figure(figsize=(6,4))
for u in roots:
    pos_scores = []
    neg_scores = []
    for x in ds_mr[u]['pos']:
        pos_scores.append(-deltaE_scores_all_qaoa(x)[u])
    for v in roots:
        if v==u: continue
        for x in ds_mr[v]['pos']:
            neg_scores.append(-deltaE_scores_all_qaoa(x)[u])
    for x in shared_neg[:min(64, len(shared_neg))]:
        neg_scores.append(-deltaE_scores_all_qaoa(x)[u])
    auc, fpr, tpr = auroc(_np.array(pos_scores), _np.array(neg_scores))
    plt.plot(fpr, tpr, label=f'root {u} AUC={auc:.2f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('RACH multi-root: one-vs-rest ROC (QAOA bank)')
plt.legend(); plt.show()

## VQE vs QAOA Banks: Side-by-Side Confusion and Per-Root AUROC

We summarize the multi-root RACH results by plotting the **VQE** and **QAOA** confusion matrices side by side and by computing an AUROC per root (one-vs-rest) for each bank.

In [ ]:
import matplotlib.pyplot as plt
import numpy as _np
import pandas as pd
from math import isnan

# Ensure roots list and bank utilities exist
try:
    roots_list = list(roots)
except NameError:
    roots_list = sorted(list(Hpos_bank.keys()))

def ensure_confusions():
    global C, Cq
    need_vqe = 'C' not in globals()
    need_qaoa = 'Cq' not in globals()
    if need_vqe:
        C_loc = _np.zeros((len(roots_list), len(roots_list)), dtype=int)
        for i,u_true in enumerate(roots_list):
            for x in ds_mr[u_true]['pos']:
                scores = {}
                for u in roots_list:
                    Ep = eV_bank(theta_bank, x, Hpos_bank[u])
                    En = eV_bank(theta_bank, x, Hneg_bank)
                    scores[u] = float(En - Ep)
                u_pred = min(scores, key=scores.get)
                C_loc[i, roots_list.index(u_pred)] += 1
        C = C_loc
    if need_qaoa:
        Cq_loc = _np.zeros((len(roots_list), len(roots_list)), dtype=int)
        for i,u_true in enumerate(roots_list):
            for x in ds_mr[u_true]['pos']:
                scores = {}
                for u in roots_list:
                    Ep = eQ_bank(params_bank, x, Hpos_bank[u])
                    En = eQ_bank(params_bank, x, Hneg_bank)
                    scores[u] = float(En - Ep)
                u_pred = min(scores, key=scores.get)
                Cq_loc[i, roots_list.index(u_pred)] += 1
        Cq = Cq_loc

ensure_confusions()

# Side-by-side confusion matrices
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(C, interpolation='nearest')
plt.title('VQE bank')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(range(len(roots_list)), roots_list); plt.yticks(range(len(roots_list)), roots_list)
for i in range(C.shape[0]):
    for j in range(C.shape[1]):
        plt.text(j, i, str(C[i,j]), ha='center', va='center')
plt.subplot(1,2,2)
plt.imshow(Cq, interpolation='nearest')
plt.title('QAOA bank')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(range(len(roots_list)), roots_list); plt.yticks(range(len(roots_list)), roots_list)
for i in range(Cq.shape[0]):
    for j in range(Cq.shape[1]):
        plt.text(j, i, str(Cq[i,j]), ha='center', va='center')
plt.tight_layout(); plt.show()

# Per-root AUROC for both banks (one-vs-rest, higher score means class membership)
def auroc(scores_pos, scores_neg):
    s = _np.concatenate([scores_pos, scores_neg])
    y = _np.concatenate([_np.ones_like(scores_pos), _np.zeros_like(scores_neg)])
    order = _np.argsort(s)
    s, y = s[order], y[order]
    uniq = _np.unique(s)
    tpr=[]; fpr=[]
    P = y.sum(); N = len(y)-P
    for t in uniq:
        pred = (s>=t).astype(int)
        TP = (pred*y).sum(); FP = (pred*(1-y)).sum()
        tpr.append(TP/(P+1e-12)); fpr.append(FP/(N+1e-12))
    order = _np.argsort(fpr)
    fpr = _np.array(fpr)[order]; tpr = _np.array(tpr)[order]
    auc = _np.trapz(tpr, fpr)
    return float(auc)

rows = []
for u in roots_list:
    # VQE bank scores
    pos_vqe=[]; neg_vqe=[]
    for x in ds_mr[u]['pos']:
        Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
        pos_vqe.append(-(float(En - Ep)))
    for v in roots_list:
        if v==u: continue
        for x in ds_mr[v]['pos']:
            Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
            neg_vqe.append(-(float(En - Ep)))
    for x in shared_neg[:min(64, len(shared_neg))]:
        Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
        neg_vqe.append(-(float(En - Ep)))
    auc_vqe = auroc(_np.array(pos_vqe), _np.array(neg_vqe))

    # QAOA bank scores
    pos_q=[]; neg_q=[]
    for x in ds_mr[u]['pos']:
        Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
        pos_q.append(-(float(En - Ep)))
    for v in roots_list:
        if v==u: continue
        for x in ds_mr[v]['pos']:
            Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
            neg_q.append(-(float(En - Ep)))
    for x in shared_neg[:min(64, len(shared_neg))]:
        Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
        neg_q.append(-(float(En - Ep)))
    auc_q = auroc(_np.array(pos_q), _np.array(neg_q))

    rows.append({'root': u, 'AUROC_VQE': auc_vqe, 'AUROC_QAOA': auc_q})

df_auc = pd.DataFrame(rows).sort_values('root')
print(df_auc)

try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user('RACH per-root AUROC (VQE vs QAOA)', df_auc)
except Exception:
    pass

## Summary: AUROC per Root (VQE vs QAOA) — Bar Chart
This chart compares VQE and QAOA banks on a per-root basis using one-vs-rest AUROC.

In [ ]:
import numpy as _np
import matplotlib.pyplot as plt
import pandas as pd

# Reuse df_auc if available; otherwise recompute from bank evaluators
need_recompute = 'df_auc' not in globals()
if need_recompute:
    def auroc(scores_pos, scores_neg):
        s = _np.concatenate([scores_pos, scores_neg])
        y = _np.concatenate([_np.ones_like(scores_pos), _np.zeros_like(scores_neg)])
        order = _np.argsort(s)
        s, y = s[order], y[order]
        uniq = _np.unique(s)
        tpr=[]; fpr=[]
        P = y.sum(); N = len(y)-P
        for t in uniq:
            pred = (s>=t).astype(int)
            TP = (pred*y).sum(); FP = (pred*(1-y)).sum()
            tpr.append(TP/(P+1e-12)); fpr.append(FP/(N+1e-12))
        order = _np.argsort(fpr)
        fpr = _np.array(fpr)[order]; tpr = _np.array(tpr)[order]
        auc = _np.trapz(tpr, fpr)
        return float(auc)
    roots_list = list(roots)
    rows = []
    for u in roots_list:
        # VQE bank
        pos_vqe=[]; neg_vqe=[]
        for x in ds_mr[u]['pos']:
            Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
            pos_vqe.append(-(float(En - Ep)))
        for v in roots_list:
            if v==u: continue
            for x in ds_mr[v]['pos']:
                Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
                neg_vqe.append(-(float(En - Ep)))
        for x in shared_neg[:min(64, len(shared_neg))]:
            Ep = eV_bank(theta_bank, x, Hpos_bank[u]); En = eV_bank(theta_bank, x, Hneg_bank)
            neg_vqe.append(-(float(En - Ep)))
        auc_vqe = auroc(_np.array(pos_vqe), _np.array(neg_vqe))
        # QAOA bank
        pos_q=[]; neg_q=[]
        for x in ds_mr[u]['pos']:
            Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
            pos_q.append(-(float(En - Ep)))
        for v in roots_list:
            if v==u: continue
            for x in ds_mr[v]['pos']:
                Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
                neg_q.append(-(float(En - Ep)))
        for x in shared_neg[:min(64, len(shared_neg))]:
            Ep = eQ_bank(params_bank, x, Hpos_bank[u]); En = eQ_bank(params_bank, x, Hneg_bank)
            neg_q.append(-(float(En - Ep)))
        auc_q = auroc(_np.array(pos_q), _np.array(neg_q))
        rows.append({'root': u, 'AUROC_VQE': auc_vqe, 'AUROC_QAOA': auc_q})
    df_auc = pd.DataFrame(rows).sort_values('root')

# Bar chart (single plot)
x = _np.arange(len(df_auc))
width = 0.4
plt.figure(figsize=(7,4))
plt.bar(x - width/2, df_auc['AUROC_VQE'], width, label='VQE')
plt.bar(x + width/2, df_auc['AUROC_QAOA'], width, label='QAOA')
plt.xticks(x, df_auc['root'].astype(str))
plt.ylim(0, 1.0)
plt.xlabel('Root')
plt.ylabel('AUROC (one-vs-rest)')
plt.title('RACH per-root AUROC: VQE vs QAOA')
plt.legend()
plt.show()

try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user('RACH per-root AUROC (table)', df_auc)
except Exception:
    pass